# Notebook de búsqueda — ADAG1 · Proyecto Oráculo

**Grupo:** G01
**Integrantes:** Andrés Santiago Restrepo Duque · Keyla Daniela Cartagena Díaz
**Curso:** Análisis y Diseño de Algoritmos · 2026-B
**Método:** ascenso por coordenadas con reinicios y **poda guiada por trazas de fallos**
**Modelo:** `Qwen/Qwen3-1.7B` (`qwen17b`), temperatura 0.0

## Qué hace este notebook

Busca una configuración de prompt (un índice por ranura `rol`, `estrategia`, `formato`, `estilo`, `cierre`, más la temperatura) que maximice la precisión del oráculo sobre las **100 instancias de búsqueda**. Deja como evidencia:

1. el registro de **todas** las configuraciones evaluadas y su precisión (`resultados/evaluaciones.csv`, `resultados/registro_busqueda.json`);
2. el análisis de trazas y las **decisiones de poda** que se tomaron con él (`resultados/poda.json`);
3. la validación final con `oraculo.validar` sobre las 87 instancias reservadas;
4. `entrega.json` con la configuración elegida.

El resultado es la **mejor configuración observada**, no un óptimo global demostrado. `oraculo.py` se usa como caja negra: solo se llaman `evaluar` y `validar`.

## Cómo ejecutarlo

1. Colab ▸ *Entorno de ejecución* ▸ *Cambiar tipo de entorno* ▸ **T4 GPU**.
2. Ejecutar la sección 1 y **reiniciar la sesión** cuando termine.
3. Ejecutar el resto de celdas de arriba hacia abajo. Si Colab se desconecta, volver a ejecutar desde la sección 2: la caché en Drive evita repetir generaciones.
4. Al final se descarga `resultados_busqueda.zip`; su contenido se sube al repositorio (ver README).

## 1 · Instalación  ·  *al terminar, reiniciar la sesión*

In [ ]:
# Librerías del modelo y de los verificadores de open-instruct.
!pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate nltk spacy emoji langdetect immutabledict
!python -m spacy download en_core_web_sm -q
!if [ ! -d /content/open-instruct ]; then git clone -q https://github.com/allenai/open-instruct /content/open-instruct; fi

# Repositorio del grupo (trae oraculo.py, ayudas.py y datos_visibles.json).
!if [ -d /content/ADA-2026-2-GP01_Andres_y_Keyla ]; then git -C /content/ADA-2026-2-GP01_Andres_y_Keyla pull -q; else git clone -q https://github.com/Keyla-Cartagena/ADA-2026-2-GP01_Andres_y_Keyla.git /content/ADA-2026-2-GP01_Andres_y_Keyla; fi

# Datos de nltk que usan los verificadores (su downloader falla con el proxy de Colab).
import io, urllib.request, zipfile
NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
for carpeta, nombre in [("tokenizers", "punkt"), ("tokenizers", "punkt_tab"),
                        ("taggers", "averaged_perceptron_tagger"),
                        ("taggers", "averaged_perceptron_tagger_eng")]:
    with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:
        zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")

print("LISTO → Entorno de ejecución ▸ Reiniciar sesión, y continuar en la sección 2")

## 2 · Parámetros, modelo, datos y oráculo

Todos los parámetros de la búsqueda están en esta celda. La caché del oráculo se guarda en Google Drive para sobrevivir a desconexiones; si no se quiere usar Drive, poner `USAR_DRIVE = False`.

In [ ]:
import os, sys, json, time, random, gzip, shutil
from pathlib import Path
from collections import Counter

RAIZ = Path("/content/ADA-2026-2-GP01_Andres_y_Keyla")
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
RESULTADOS = RAIZ / "resultados"
RESULTADOS.mkdir(exist_ok=True)

# ---- parámetros de la búsqueda -------------------------------------------
GRUPO = "G01"
SEMANA = 5            # confirmar con la profesora antes de entregar
MODELO = "qwen17b"
TEMPERATURA = 0.0     # greedy: misma config + misma instancia → misma respuesta
SEMILLA = 1           # semilla del oráculo y de los puntos iniciales aleatorios
MAX_PASADAS = 3       # pasadas máximas por reinicio
N_REINICIOS = 3       # 1 inicio en ceros + 2 inicios aleatorios reproducibles
UMBRAL_PODA = 3       # pérdida neta (perdidas − ganadas) que cuenta como derrota
CONTEXTOS_PODA = 2    # contextos distintos con derrota necesarios para podar
MIN_VIVAS = 4         # opciones mínimas por ranura: nunca se poda más de la mitad
USAR_DRIVE = True

if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA_CACHE = Path("/content/drive/MyDrive/ADA_Oraculo_G01")
else:
    CARPETA_CACHE = RESULTADOS
CARPETA_CACHE.mkdir(parents=True, exist_ok=True)
RUTA_CACHE = CARPETA_CACHE / "cache_oraculo.json"
print("caché:", RUTA_CACHE)

In [ ]:
from ayudas import cargar_modelo, cargar_datos, dividir, curva, entrega
from oraculo import Oraculo, CATALOGO, RANURAS, espacio

modelo = cargar_modelo(MODELO)
datos = cargar_datos(str(RAIZ / "datos_visibles.json"))
busqueda, validacion = dividir(datos)
oraculo = Oraculo(modelo, busqueda, validacion, cache=str(RUTA_CACHE),
                  lote=16, presupuesto=32_000_000)

OPCIONES = {r: list(range(len(CATALOGO[r]))) for r in RANURAS}
INSTANCIAS = busqueda

assert len(busqueda) == 100 and len(validacion) == 87
assert all(len(v) == 8 for v in OPCIONES.values())
print("espacio con temperatura fija:", len(espacio()), "configuraciones")

## 3 · Registro de evaluaciones

`evaluar_registrada` envuelve `oraculo.evaluar` y guarda, por cada configuración: precisión, aciertos, **los ids de todas las instancias falladas** y la familia que violó cada una (`violo`). Con eso se puede comparar instancia por instancia qué rompe cada opción.

Una configuración que ya se evaluó no vuelve a contarse como consulta nueva (`memo`); el oráculo además reutiliza su caché en disco.

In [ ]:
registro = []      # una fila por consulta nueva al oráculo
memo = {}          # clave de config -> resumen de su evaluación
historial = []     # precisión de cada consulta nueva, para curva()

def clave(c):
    return tuple(c[r] for r in RANURAS) + (c["temperatura"],)

def evaluar_registrada(config, **meta):
    k = clave(config)
    if k in memo:
        return memo[k]
    t0 = time.perf_counter()
    r = oraculo.evaluar(config, INSTANCIAS, semilla=SEMILLA)
    segundos = time.perf_counter() - t0
    fallos = {t["id"]: t["violo"] for t in r.trazas}
    info = {
        "config": dict(config),
        "precision": float(r.precision),
        "aciertos": r.n - len(r.trazas),
        "n": r.n,
        "fallos": fallos,                       # id de instancia -> familia violada
        "familias": Counter(fallos.values()),
        "trazas": r.trazas,                     # salidas completas, solo en memoria
    }
    memo[k] = info
    historial.append(info["precision"])
    registro.append({
        "consulta": len(registro) + 1,
        "config": dict(config),
        "precision": info["precision"],
        "aciertos": info["aciertos"],
        "n": r.n,
        "segundos": round(segundos, 2),
        "familias_violadas": dict(info["familias"]),
        "ids_fallidos": sorted(fallos),
        **meta,
    })
    mejor = max(historial)
    print(f"consulta {len(registro):3d}  {info['precision']:6.1%}  mejor {mejor:6.1%}  "
          f"{meta.get('ranura', '')}={meta.get('opcion', '')}  {segundos:5.1f}s", flush=True)
    return info

## 4 · Heurística: ascenso por coordenadas con poda guiada por trazas

**Paso básico.** Con todas las ranuras fijas menos una, se evalúan las opciones *vivas* de esa ranura y se queda la mejor. Una pasada recorre `rol → estrategia → formato → estilo → cierre`. Un reinicio termina cuando una pasada no mejora o se alcanza `MAX_PASADAS`.

**Uso de las trazas (poda).** Tras evaluar una ranura se compara cada opción `o` con la mejor `b` **instancia por instancia**:

- `perdidas(o)`: instancias que `b` resuelve y `o` falla; `ganadas(o)`: instancias que `o` resuelve y `b` falla.
- Si `|perdidas(o)| − |ganadas(o)| ≥ UMBRAL_PODA`, se anota una **derrota** de `o` en ese *contexto* (los índices de las otras cuatro ranuras), junto con las familias que `o` violó en sus instancias perdidas.
- `o` se **descarta** de la ranura solo cuando acumula derrotas en `CONTEXTOS_PODA` contextos distintos. Una sola derrota no basta: las ranuras interactúan y una opción mala con un vecindario puede ser buena con otro.
- Nunca se dejan menos de `MIN_VIVAS` opciones por ranura.

**Desempate.** Si varias opciones empatan en precisión, se conserva la actual (no se cambia sin evidencia).

**Costo.** Una pasada evalúa a lo sumo Σ kⱼ = 40 configuraciones de 100 instancias; con poda, cada pasada posterior evalúa solo las opciones vivas. Cota: O(R · I · m · K · n) consultas, con R reinicios, I pasadas, m = 5 ranuras, K = 8 opciones y n = 100 instancias.

In [ ]:
vivas = {r: list(OPCIONES[r]) for r in RANURAS}   # opciones que siguen en juego
podas = []                                        # decisiones tomadas con las trazas
derrotas = {}                                     # (ranura, opción) -> {contexto: evidencia}
TODAS = set(x["id"] for x in INSTANCIAS)

def podar(ranura, resultados, mejor_opcion, actual, contexto):
    b = resultados[mejor_opcion]
    resueltas_b = TODAS - set(b["fallos"])
    vecindario = tuple(actual[r] for r in RANURAS if r != ranura)
    # 1) anotar derrotas con evidencia instancia por instancia
    for o, info in resultados.items():
        if o == mejor_opcion:
            continue
        perdidas = sorted(i for i in resueltas_b if i in info["fallos"])
        ganadas = sorted(i for i in (TODAS - set(info["fallos"])) if i in b["fallos"])
        if len(perdidas) - len(ganadas) >= UMBRAL_PODA:
            derrotas.setdefault((ranura, o), {})[vecindario] = {
                "frente_a": mejor_opcion, "perdidas": perdidas, "ganadas": ganadas,
                "familias": Counter(info["fallos"][i] for i in perdidas),
                "precision": info["precision"], "precision_mejor": b["precision"],
                "info": info, **contexto,
            }
    # 2) podar las opciones derrotadas en suficientes contextos, de la que más pierde a la que menos
    candidatas = []
    for (r, o), ev in derrotas.items():
        if r == ranura and o != mejor_opcion and o in vivas[ranura] and len(ev) >= CONTEXTOS_PODA:
            neto = sum(len(e["perdidas"]) - len(e["ganadas"]) for e in ev.values())
            candidatas.append((neto, o, ev))
    candidatas.sort(key=lambda x: (-x[0], x[1]))
    for neto, o, ev in candidatas:
        if len(vivas[ranura]) <= MIN_VIVAS:
            break
        vivas[ranura].remove(o)
        familias = sum((e["familias"] for e in ev.values()), Counter())
        ultima = list(ev.values())[-1]
        ejemplo = next(t for t in ultima["info"]["trazas"] if t["id"] == ultima["perdidas"][0])
        podas.append({
            "ranura": ranura,
            "opcion_podada": o,
            "contextos": len(ev),
            "perdida_neta_total": neto,
            "evidencia": [{"vecindario": list(v), "frente_a": e["frente_a"],
                           "precision_podada": e["precision"], "precision_mejor": e["precision_mejor"],
                           "instancias_perdidas": e["perdidas"], "instancias_ganadas": e["ganadas"],
                           "reinicio": e["reinicio"], "pasada": e["pasada"]} for v, e in ev.items()],
            "familias_en_perdidas": dict(familias.most_common()),
            "ejemplo_id": ejemplo["id"],
            "ejemplo_salida_inicio": ejemplo["salida"][:160],
            "ejemplo_salida_final": ejemplo["salida"][-120:],
            **contexto,
        })
        print(f"  poda {ranura}={o}: derrotada en {len(ev)} contextos, pérdida neta {neto} "
              f"instancias; familias {dict(familias.most_common(3))}")

def ascenso_por_coordenadas(inicial, reinicio):
    actual = dict(inicial)
    p_actual = evaluar_registrada(actual, reinicio=reinicio, pasada=0,
                                  ranura="inicial", opcion=None)["precision"]
    for pasada in range(1, MAX_PASADAS + 1):
        mejoro = False
        for ranura in RANURAS:
            opciones = sorted(set(vivas[ranura]) | {actual[ranura]})
            resultados = {}
            for o in opciones:
                c = dict(actual, **{ranura: o})
                resultados[o] = evaluar_registrada(c, reinicio=reinicio, pasada=pasada,
                                                   ranura=ranura, opcion=o)
            mejor_p = max(info["precision"] for info in resultados.values())
            mejor_o = actual[ranura] if resultados[actual[ranura]]["precision"] == mejor_p \
                else min(o for o, info in resultados.items() if info["precision"] == mejor_p)
            podar(ranura, resultados, mejor_o, actual, {"reinicio": reinicio, "pasada": pasada})
            if mejor_p > p_actual:
                mejoro = True
            actual[ranura], p_actual = mejor_o, max(p_actual, mejor_p)
        print(f"fin pasada {pasada} (reinicio {reinicio}): {p_actual:.1%}  {actual}")
        if not mejoro:
            break
    return actual, p_actual

### Ejecución de la búsqueda

Puntos iniciales reproducibles: todos los índices en 0 y `N_REINICIOS − 1` configuraciones aleatorias generadas con `SEMILLA`. Ninguno usa conocimiento previo de qué opción es buena.

In [ ]:
rng = random.Random(SEMILLA)
iniciales = [{r: 0 for r in RANURAS} | {"temperatura": TEMPERATURA}]
while len(iniciales) < N_REINICIOS:
    iniciales.append({r: rng.randrange(8) for r in RANURAS} | {"temperatura": TEMPERATURA})

t_inicio = time.time()
reinicios = []
for numero, inicial in enumerate(iniciales, start=1):
    print(f"\n===== reinicio {numero}: inicio {inicial} =====")
    config, precision = ascenso_por_coordenadas(inicial, numero)
    reinicios.append({"reinicio": numero, "inicial": inicial,
                      "config": config, "precision": precision})

mejor = max(reinicios, key=lambda x: x["precision"])
MEJOR_CONFIG, MEJOR_PRECISION = mejor["config"], mejor["precision"]
print("\nmejor configuración observada:", MEJOR_CONFIG)
print(f"precisión en búsqueda: {MEJOR_PRECISION:.1%}")
print("consultas nuevas al oráculo:", len(registro))
print(f"tiempo total: {(time.time() - t_inicio) / 60:.1f} min")
curva(historial)

## 5 · Registro de evaluaciones (tabla)

In [ ]:
import pandas as pd

tabla = pd.DataFrame([{
    "consulta": f["consulta"], "reinicio": f["reinicio"], "pasada": f["pasada"],
    "ranura": f["ranura"], "opcion": f["opcion"],
    **{r: f["config"][r] for r in RANURAS},
    "precision": f["precision"], "aciertos": f["aciertos"], "segundos": f["segundos"],
} for f in registro])
tabla.to_csv(RESULTADOS / "evaluaciones.csv", index=False)
pd.set_option("display.max_rows", 400)
tabla

In [ ]:
# Resumen por reinicio y top 10 de configuraciones evaluadas.
display(pd.DataFrame(reinicios))
tabla.sort_values("precision", ascending=False).head(10)

## 6 · Análisis de trazas y decisiones derivadas

### 6.1 Opciones podadas y por qué

Cada fila es una decisión: qué opción se descartó, frente a cuál, cuántas instancias perdía y **qué familias de restricciones rompió** en esas instancias. La familia indica la superficie de la respuesta que la opción altera (por ejemplo `startend:*` o `first_word:*` → prefijo; `length_constraints:*` → largo; `paragraphs:*`/`detectable_format:*` → estructura; `change_case:*`/`punctuation:*` → caracteres; `last_word:*`/`startend:end_checker` → sufijo).

In [ ]:
tabla_podas = pd.DataFrame([{
    "ranura": p["ranura"], "podada": p["opcion_podada"], "frente_a": p["frente_a"],
    "prec_podada": p["precision_podada"], "prec_mejor": p["precision_mejor"],
    "perdidas": len(p["instancias_perdidas"]),
    "familias": ", ".join(f"{k}×{v}" for k, v in list(p["familias_en_perdidas"].items())[:3]),
    "reinicio": p["reinicio"], "pasada": p["pasada"],
} for p in podas])
print("opciones vivas al final:", vivas)
tabla_podas

### 6.2 Evidencia textual de una opción podada por ranura

In [ ]:
vistas = set()
for p in podas:
    if p["ranura"] in vistas:
        continue
    vistas.add(p["ranura"])
    print(f"--- {p['ranura']}={p['opcion_podada']}  (instancia {p['ejemplo_id']})")
    print("texto de la opción:", CATALOGO[p["ranura"]][p["opcion_podada"]][:140])
    print("inicio de la salida:", repr(p["ejemplo_salida_inicio"]))
    print("final de la salida: ", repr(p["ejemplo_salida_final"]))
    print("familias violadas en sus instancias perdidas:", p["familias_en_perdidas"], "\n")

### 6.3 Fallos de la configuración elegida

Las familias que todavía tumban a la mejor configuración muestran qué restricciones ninguna opción viva logró arreglar.

In [ ]:
info_mejor = memo[clave(MEJOR_CONFIG)]
print(f"{info_mejor['aciertos']}/{info_mejor['n']} instancias resueltas")
pd.Series(info_mejor["familias"]).sort_values(ascending=False).head(15)

### 6.4 Cómo cambió la búsqueda por las trazas

La celda calcula cuántas consultas se ahorraron gracias a la poda: sin poda, cada pasada de cada reinicio evaluaría las 8 opciones de las 5 ranuras.

In [ ]:
sin_poda = sum(1 + 40 * max(f["pasada"] for f in registro if f["reinicio"] == r["reinicio"])
               for r in reinicios)
print("consultas que habría hecho el ascenso sin poda:", sin_poda)
print("consultas nuevas realizadas con poda:          ", len(registro))
print("opciones descartadas por evidencia de trazas:  ", len(podas))

## 7 · Validación final con `oraculo.validar`

Se mide la configuración elegida sobre las **87 instancias de validación**, que no se usaron para buscar. Esta medición **no** se usa para cambiar la configuración. Tampoco es la evaluación oficial: la profesora califica con datos de prueba privados.

In [ ]:
r_val = oraculo.validar(MEJOR_CONFIG, n=None)
PRECISION_VALIDACION = float(r_val.precision)
print(f"búsqueda:   {MEJOR_PRECISION:.1%}  ({info_mejor['aciertos']}/{info_mejor['n']})")
print(f"validación: {PRECISION_VALIDACION:.1%}  ({r_val.n - len(r_val.trazas)}/{r_val.n})")

## 8 · Guardar evidencia y `entrega.json`

In [ ]:
with open(RESULTADOS / "registro_busqueda.json", "w", encoding="utf-8") as f:
    json.dump({
        "grupo": GRUPO, "modelo": MODELO, "temperatura": TEMPERATURA, "semilla": SEMILLA,
        "metodo": "ascenso por coordenadas con reinicios y poda guiada por trazas",
        "parametros": {"max_pasadas": MAX_PASADAS, "reinicios": N_REINICIOS,
                       "umbral_poda": UMBRAL_PODA, "min_vivas": MIN_VIVAS},
        "consultas_nuevas": len(registro),
        "reinicios": reinicios,
        "mejor_config": MEJOR_CONFIG,
        "precision_busqueda": MEJOR_PRECISION,
        "precision_validacion": PRECISION_VALIDACION,
        "fallos_validacion": dict(Counter(t["violo"] for t in r_val.trazas)),
        "opciones_vivas_final": vivas,
        "evaluaciones": registro,
    }, f, ensure_ascii=False, indent=1)

with open(RESULTADOS / "poda.json", "w", encoding="utf-8") as f:
    json.dump(podas, f, ensure_ascii=False, indent=1)

import matplotlib.pyplot as plt
acumulado = pd.Series(historial).cummax()
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(historial) + 1), historial, ".", alpha=0.4, label="consulta")
plt.plot(range(1, len(historial) + 1), acumulado, label="mejor hasta ahora")
plt.xlabel("consulta nueva al oráculo"); plt.ylabel("precisión en búsqueda"); plt.legend()
plt.savefig(RESULTADOS / "curva_busqueda.png", dpi=150, bbox_inches="tight")
plt.show()

# entrega.json en la raíz del repositorio: es el que carga el notebook de evaluación.
entrega(GRUPO, MEJOR_CONFIG, SEMANA, ruta=str(RAIZ / "entrega.json"))

# Copia comprimida de la caché: permite rehacer el análisis sin GPU.
with open(RUTA_CACHE, "rb") as fi, gzip.open(RESULTADOS / "cache_oraculo.json.gz", "wb") as fo:
    shutil.copyfileobj(fi, fo)

In [ ]:
# Descargar todo lo que hay que subir al repositorio.
paquete = Path("/content/resultados_busqueda")
if paquete.exists():
    shutil.rmtree(paquete)
shutil.copytree(RESULTADOS, paquete / "resultados")
shutil.copy(RAIZ / "entrega.json", paquete / "entrega.json")
shutil.make_archive("/content/resultados_busqueda", "zip", paquete)

from google.colab import files
files.download("/content/resultados_busqueda.zip")
print("Además: Archivo ▸ Descargar ▸ .ipynb, para subir este notebook con sus salidas.")